In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_table = 'workspace.taxi.bronze_trips'
silver_table = 'workspace.taxi.silver_trips'

In [0]:
df = spark.table(bronze_table)

In [0]:
df.display()

In [0]:
silver_df = df.dropDuplicates(['VendorID','tpep_dropoff_datetime','tpep_pickup_datetime'])\
    .withColumn('pickup_ts',to_timestamp('tpep_pickup_datetime'))\
    .withColumn('dropoff_ts',to_timestamp('tpep_dropoff_datetime'))\
    .withColumn('trip_distance',col('trip_distance').cast('double'))\
    .withColumn('fare_amount',col('fare_amount').cast('double'))\
    .withColumn('passenger_count',col('passenger_count').cast('integer'))\
    .filter(col('fare_amount') > 0)\
    .filter(col('trip_distance') > 0)\
    .filter(col('pickup_ts').isNotNull())\
    .select('vendorID','pickup_ts','dropoff_ts','passenger_count','trip_distance','fare_amount','payment_type','PULocationID','DOLocationID')

In [0]:
silver_df.display()

In [0]:
silver_df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(silver_table)

print(f"silver transformation complete Rows: {spark.table(silver_table).count()}")